In [34]:
# TASK #1
# Convert the following searching algorithms into agent-based models:
# ● DLS: Implement as a Goal-Based Agent to explore the graph up to a specified depth limit.

class Environment:
    def __init__(self, graph):
        self.graph = graph

    def get_percept(self, node):
        return node

class GoalBasedAgent:
    def __init__(self, goal, depth_limit):
        self.goal = goal
        self.depth_limit = depth_limit

    def formulate_goal(self, percept):
        if percept == self.goal:
            return "Goal reached"
        return "Searching"

    def dls(self, graph, node, depth, visited):
        if depth > self.depth_limit:
            return False  # Limit reached
        
        visited.append(node)
        if node == self.goal: return True
        
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                if self.dls(graph, neighbor, depth+1, visited):
                    return True
            
        visited.pop()  # Backtrack if goal not found
        return False

    def act(self, percept, graph):
        goal_status = self.formulate_goal(percept)
        if goal_status == "Goal reached":
            return f"Goal {self.goal} found!"
        else:
            visited = []
            if self.dls(graph, percept, 0, visited):
                return f"Goal found with DLS Goal Based Agent. \nPath: {visited}"
            else:
                return f"Goal not found within the depth limit."


def run_agent(agent, environment, start_node):
    percept = environment.get_percept(start_node)
    action = agent.act(percept, environment.graph)  # Pass graph to agent
    print(action)


# Tree Representation
tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H'],
    'E': [],
    'F': ['I'],
    'G': [],
    'H': [],
    'I': []
}

# Define Start and Goal Nodes
start_node = 'A'
goal_node = 'I'

# Create instances of agent and environment
agent = GoalBasedAgent(goal_node, depth_limit=3)
environment = Environment(tree)

# Run the agent
run_agent(agent, environment, start_node)


Goal found with DLS Goal Based Agent. 
Path: ['A', 'C', 'F', 'I']


In [1]:
class MazeSolver:
    def __init__(self, maze):
        self.maze = maze
        self.rows = len(maze)
        self.cols = len(maze[0])
        self.start = self.find_position('S')
        self.goal = self.find_position('G')
        
    def find_position(self, symbol):
        """Find the coordinates of a symbol in the maze"""
        for i in range(self.rows):
            for j in range(self.cols):
                if self.maze[i][j] == symbol:
                    return (i, j)
        return None
    
    def is_valid_move(self, position):
        """Check if a move is valid (within bounds and not an obstacle)"""
        row, col = position
        if row < 0 or row >= self.rows or col < 0 or col >= self.cols:
            return False
        # Check if the cell is not an obstacle (1) and is within bounds
        return self.maze[row][col] != 1
    
    def get_neighbors(self, position):
        """Get valid neighboring positions"""
        row, col = position
        # Define possible moves: up, down, left, right
        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        neighbors = []
        
        for dr, dc in moves:
            new_pos = (row + dr, col + dc)
            if self.is_valid_move(new_pos):
                neighbors.append(new_pos)
        
        return neighbors
    
    def depth_limited_search(self, node, goal, depth, path, visited):
        """
        Depth-Limited Search implementation
        Returns: (found, path)
        """
        if node == goal:
            return True, path + [node]
        
        if depth <= 0:
            return False, None
        
        # Mark current node as visited for this path
        visited.add(node)
        
        # Explore neighbors
        for neighbor in self.get_neighbors(node):
            if neighbor not in visited:
                found, result_path = self.depth_limited_search(
                    neighbor, goal, depth - 1, path + [node], visited
                )
                if found:
                    return True, result_path
        
        # Backtrack
        visited.remove(node)
        return False, None
    
    def dfs_with_limit(self, max_depth):
        """
        Depth-First Search with depth limit
        Returns: (found, path, nodes_expanded)
        """
        visited = set()
        nodes_expanded = 0
        
        def dfs(node, goal, depth, path, visited):
            nonlocal nodes_expanded
            nodes_expanded += 1
            
            if node == goal:
                return True, path + [node]
            
            if depth <= 0:
                return False, None
            
            visited.add(node)
            
            for neighbor in self.get_neighbors(node):
                if neighbor not in visited:
                    found, result_path = dfs(neighbor, goal, depth - 1, path + [node], visited)
                    if found:
                        return True, result_path
            
            visited.remove(node)
            return False, None
        
        return dfs(self.start, self.goal, max_depth, [], visited), nodes_expanded
    
    def iterative_deepening_search(self):
        """
        Iterative Deepening Search implementation
        Returns: (found, path, nodes_expanded, final_depth)
        """
        depth = 0
        total_nodes_expanded = 0
        
        while True:
            visited = set()
            nodes_expanded_this_depth = 0
            
            def dls(node, goal, current_depth, path, visited):
                nonlocal nodes_expanded_this_depth
                nodes_expanded_this_depth += 1
                
                if node == goal:
                    return True, path + [node]
                
                if current_depth <= 0:
                    return False, None
                
                visited.add(node)
                
                for neighbor in self.get_neighbors(node):
                    if neighbor not in visited:
                        found, result_path = dls(neighbor, goal, current_depth - 1, path + [node], visited)
                        if found:
                            return True, result_path
                
                visited.remove(node)
                return False, None
            
            found, path = dls(self.start, self.goal, depth, [], visited)
            total_nodes_expanded += nodes_expanded_this_depth
            
            if found:
                return True, path, total_nodes_expanded, depth
            
            depth += 1
            
            # Safety check to prevent infinite loop (in case goal is unreachable)
            if depth > self.rows * self.cols:
                return False, None, total_nodes_expanded, depth
    
    def print_maze_with_path(self, path):
        """Print the maze with the path marked"""
        if not path:
            print("No path found")
            return
        
        # Create a copy of the maze for visualization
        visual_maze = []
        for i in range(self.rows):
            row = []
            for j in range(self.cols):
                if self.maze[i][j] == 'S':
                    row.append('S')
                elif self.maze[i][j] == 'G':
                    row.append('G')
                elif self.maze[i][j] == 1:
                    row.append('█')  # Obstacle
                else:
                    row.append('.')  # Empty cell
            visual_maze.append(row)
        
        # Mark the path (excluding start and goal)
        for pos in path[1:-1]:
            i, j = pos
            if visual_maze[i][j] == '.':
                visual_maze[i][j] = '*'
        
        # Print the maze
        print("Maze with path (* = path):")
        for row in visual_maze:
            print(' '.join(str(cell) for cell in row))
        print()


def main():
    # Define the maze
    maze = [
        ['S', 0, 0, 1, 0],
        [1, 0, 1, 0, 0],
        [0, 0, 0, 0, 'G'],
        [1, 1, 0, 1, 1]
    ]
    
    # Create solver instance
    solver = MazeSolver(maze)
    
    print("=" * 50)
    print("MAZE SOLVER - DFS with Depth Limit and IDS")
    print("=" * 50)
    print(f"Start position: {solver.start}")
    print(f"Goal position: {solver.goal}")
    print()
    
    # Test Depth-First Search with different depth limits
    print("DEPTH-FIRST SEARCH WITH DEPTH LIMIT")
    print("-" * 40)
    
    depth_limits = [3, 5, 7, 10]
    for limit in depth_limits:
        print(f"\nDepth limit: {limit}")
        (found, path), nodes_expanded = solver.dfs_with_limit(limit)
        
        if found:
            print(f"✓ Path found!")
            print(f"  Path length: {len(path)}")
            print(f"  Nodes expanded: {nodes_expanded}")
            print(f"  Path: {path}")
            solver.print_maze_with_path(path)
        else:
            print(f"✗ No path found within depth limit {limit}")
            print(f"  Nodes expanded: {nodes_expanded}")
    
    # Test Iterative Deepening Search
    print("\n" + "=" * 50)
    print("ITERATIVE DEEPENING SEARCH")
    print("-" * 40)
    
    found, path, nodes_expanded, final_depth = solver.iterative_deepening_search()
    
    if found:
        print(f"\n✓ Path found!")
        print(f"  Final depth: {final_depth}")
        print(f"  Path length: {len(path)}")
        print(f"  Total nodes expanded across all iterations: {nodes_expanded}")
        print(f"  Path: {path}")
        solver.print_maze_with_path(path)
        
        # Verify the path is valid
        print("Path verification:")
        for i, pos in enumerate(path[:-1]):
            next_pos = path[i + 1]
            neighbors = solver.get_neighbors(pos)
            if next_pos in neighbors:
                print(f"  Step {i+1}: {pos} -> {next_pos} ✓")
            else:
                print(f"  Step {i+1}: {pos} -> {next_pos} ✗ (invalid move)")
    else:
        print("✗ No path found - goal might be unreachable")


if __name__ == "__main__":
    main()

MAZE SOLVER - DFS with Depth Limit and IDS
Start position: (0, 0)
Goal position: (2, 4)

DEPTH-FIRST SEARCH WITH DEPTH LIMIT
----------------------------------------

Depth limit: 3
✗ No path found within depth limit 3
  Nodes expanded: 5

Depth limit: 5
✗ No path found within depth limit 5
  Nodes expanded: 9

Depth limit: 7
✓ Path found!
  Path length: 7
  Nodes expanded: 11
  Path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (2, 4)]
Maze with path (* = path):
S * . █ .
█ * █ . .
. * * * G
█ █ . █ █


Depth limit: 10
✓ Path found!
  Path length: 9
  Nodes expanded: 12
  Path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (1, 3), (1, 4), (2, 4)]
Maze with path (* = path):
S * . █ .
█ * █ * *
. * * * G
█ █ . █ █


ITERATIVE DEEPENING SEARCH
----------------------------------------

✓ Path found!
  Final depth: 6
  Path length: 7
  Total nodes expanded across all iterations: 38
  Path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (2, 4)]
Maze with path (* = path):
S * . █ .


In [33]:
# TASK #1
# Convert the following searching algorithms into agent-based models:
# ● UCS: Implement as a Utility-Based Agent to find the goal with the minimum cost path.
import heapq

class Environment:
    def __init__(self, graph):
        self.graph = graph

    def get_percept(self, node):
        return node
    
class UtilityBasedAgent:
    def __init__(self, goal):
        self.goal = goal
    
    def act(self, percept, environment):
        # Agent chooses action based on utility (minimum cost path)
        return self.ucs_search(environment.graph, percept)

    def ucs_search(self, graph, start_node):
        frontier = []  # Priority queue (cost, node)
        heapq.heappush(frontier, (0, start_node, [start_node]))
        visited = set()

        while frontier:
            current_cost, current_node, path = heapq.heappop(frontier)

            if current_node == self.goal:
                return path, current_cost

            if current_node in visited:
                continue

            visited.add(current_node)

            for neighbor, cost in graph.get(current_node, {}).items():
                if neighbor not in visited:
                    heapq.heappush(frontier, (cost+current_cost, neighbor, path+[neighbor]))

        return None, None


def run_agent(agent, environment, start_node):
    percept = environment.get_percept(start_node)
    path, total_cost = agent.act(percept, environment)
    
    if path:
        print(f"Goal reached \nPath: {path} \nTotal cost: {total_cost}")
    else:
        print("Goal not reachable from the start node.")


# Graph Representation with Costs
graph = {
    'A': {'B': 2, 'C': 1},
    'B': {'D': 4, 'E': 3},
    'C': {'F': 1, 'G': 5},
    'D': {'H': 2},
    'E': {},
    'F': {'I': 6},
    'G': {},
    'H': {},
    'I': {}
}

start_node = 'A'
goal_node = 'I'

environment = Environment(graph)
agent = UtilityBasedAgent(goal_node)

run_agent(agent, environment, start_node)

Goal reached 
Path: ['A', 'C', 'F', 'I'] 
Total cost: 8


In [32]:
# TASK # 2 : Traveling Salesman Problem:
# Given a set of cities and distances between every pair of cities, the problem is to find the shortest possible route that visits every city exactly once and returns to the starting point. 
# Like any problem, which can be optimized, there must be a cost function.  In the context of TSP, total distance traveled must be reduced as much as possible.
# Consider the below matrix representing the distances (Cost) between the cities. 
# Find the shortest possible route that visits every city exactly once and returns to the starting point.

import itertools    # provides fast tools for working with sequences and combinations.

class Environment:
    def __init__(self, distance_dict):
        self.distance_dict = distance_dict

    def get_percept(self):
        return self.distance_dict

class UtilityBasedAgent:
    def __init__(self, start_city):
        self.start_city = start_city

    def act(self, percept):
        return self.solve_tsp(percept)

    def solve_tsp(self, dist):
        cities = list(dist.keys())
        min_cost = float('inf')
        best_route = None

        # Remove start city from permutation list
        other_cities = [city for city in cities if city != self.start_city]

        print("All Possible Routes:\n")

        for perm in itertools.permutations(other_cities):
            route = [self.start_city] + list(perm) + [self.start_city]

            cost = self.calculate_cost(route, dist)

            print("Route:", " -> ".join(map(str, route)),
                  "| Cost:", cost)

            if cost < min_cost:
                min_cost = cost
                best_route = route

        return best_route, min_cost

    def calculate_cost(self, route, dist):
        total_cost = 0
        for i in range(len(route) - 1):
            total_cost += dist[route[i]][route[i + 1]]
        return total_cost

def run_agent(agent, environment):
    percept = environment.get_percept()
    best_route, min_cost = agent.act(percept)

    print("\nOptimal Route Found!")
    print("Best Route:", " -> ".join(map(str, best_route)))
    print("Minimum Cost:", min_cost)

dist = {
    1: {2: 15, 3: 45, 4: 30},
    2: {1: 20, 3: 35, 4: 45},
    3: {1: 35, 2: 25, 4: 30},
    4: {1: 20, 2: 5, 3: 30}
}

start_city = 1

environment = Environment(dist)
agent = UtilityBasedAgent(start_city)

run_agent(agent, environment)

All Possible Routes:

Route: 1 -> 2 -> 3 -> 4 -> 1 | Cost: 100
Route: 1 -> 2 -> 4 -> 3 -> 1 | Cost: 125
Route: 1 -> 3 -> 2 -> 4 -> 1 | Cost: 135
Route: 1 -> 3 -> 4 -> 2 -> 1 | Cost: 100
Route: 1 -> 4 -> 2 -> 3 -> 1 | Cost: 105
Route: 1 -> 4 -> 3 -> 2 -> 1 | Cost: 105

Optimal Route Found!
Best Route: 1 -> 2 -> 3 -> 4 -> 1
Minimum Cost: 100


In [ ]:
# TASK # 3 : Implement Iterative deepening DFS on graph and tree.

class Environment:
    def __init__(self, structure):
        self.structure = structure  # Can be tree or graph

    def get_percept(self, start_node):
        return start_node

class IDDFSAgent:
    def __init__(self, goal, max_depth=10):
        self.goal = goal
        self.max_depth = max_depth

    def dls(self, graph, node, depth, path):
        path.append(node)

        if node == self.goal:
            return True

        if depth <= 0:
            path.pop()
            return False

        for neighbor in graph.get(node, []):
            if neighbor not in path:
                if self.dls(graph, neighbor, depth - 1, path):
                    return True

        path.pop()
        return False

    def iddfs(self, graph, start_node):
        for depth in range(self.max_depth + 1):
            path = []
            if self.dls(graph, start_node, depth, path):
                return path, depth
        return None, None

def run_agent(agent, environment, start):
    percept = environment.get_percept(start)
    path, depth = agent.iddfs(environment.structure, percept)

    if path:
        formatted_path = [str(node) for node in path]
        print(f"Goal found \nPath: {formatted_path} \nDepth reached: {depth}")
    else:
        print("Goal not found within depth limit.")

print("Iterative deepening DFS on graph: \n")

graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': [],
    'E': [],
    'F': [],
    'G': []
}

environment = Environment(graph)
agent = IDDFSAgent('F', max_depth=5)

run_agent(agent, environment, 'A')

print("\nIterative deepening DFS on Tree:\n")

tree = {
    1: [2, 3],
    2: [4, 5],
    3: [6, 7],
    4: [],
    5: [],
    6: [],
    7: []
}

environment = Environment(tree)
agent = IDDFSAgent(6, max_depth=5)

run_agent(agent, environment, 1)

Iterative deepening DFS on graph: 

Goal found 
Path: ['A', 'C', 'F'] 
Depth reached: 2

Iterative deepening DFS on Tree:

Goal found 
Path: ['1', '3', '6'] 
Depth reached: 2
